# dim_dish_scd1 (SCD 1, poc method)

One row per `restaurant_id` × `dish_name_normalized`: dishes the YouTube reviews name (`stg_restaurant_mentions.dishes`, extraction v2), child of `dim_restaurant_scd2`. Same content as `dim_dish`.

- `price_vnd` = median price said (5,000–5,000,000 VND), `is_signature` = a review called it the signature dish, `video_count`, `first_seen_at` / `last_seen_at`.
- **Source is rebuilt in full every run** (a dish aggregates every mention of the restaurant; ~2,000 rows). MERGE on `primary_checksum`: update when `record_checksum` differs, insert new dishes, delete dishes no longer built (restaurant id changed).

In [ ]:
%run ./_udfs

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
path = f"{gold}.dim_dish_scd1"
if not spark.catalog.tableExists(path):
    load_type = "full"
print(f"load_type = {load_type}")
GEOCODE = f"{silver}.stg_restaurant_geocode"

# Read Source Table

Every mention with its durable `restaurant_id` (same rule as `dim_restaurant_scd2`).

In [ ]:
# --- Every mention with its durable restaurant_id (dedup rule above; NULL until geocode matched it) ---
spark.sql(rf"""
    CREATE OR REPLACE TEMP VIEW mentions_keyed AS
    WITH m AS (
        SELECT *,
            nullif(trim(regexp_replace(regexp_replace(regexp_replace(
                lower(unaccent_vi(coalesce(district, ''))),
                '^(q\\.?\\s*|quan\\s+)0*(\\d+)$', 'quan $2'),
                '^(tp\\.?|thanh pho)\\s+', ''),
                '\\s+', ' ')), '') AS district_norm
        FROM {silver}.stg_restaurant_mentions
    )
    SELECT m.*,
           g.place_id, g.place_name, g.formatted_address, g.ward AS geo_ward, g.city AS geo_city,
           g.latitude, g.longitude,
           -- durable id from the Google place (same place = same restaurant); a restaurant only located by
           -- Photon (osm_approx, no place_id, 2026-11-03) is keyed by name + district until a Google place
           -- matches it (then it gets the place id, the old id is closed); NULL = not located yet
           CASE WHEN g.place_id IS NOT NULL THEN md5('place:' || g.place_id)
                WHEN g.match_status = 'osm_approx'
                THEN md5('name:' || m.restaurant_name_normalized || '|' || coalesce(m.district_norm, '')) END AS restaurant_id
    FROM m
    LEFT JOIN {GEOCODE} g
      ON g.restaurant_name_normalized = m.restaurant_name_normalized AND g.district_norm <=> m.district_norm
     AND g.match_status IN ('matched', 'places_match', 'osm_approx')
""")
has_dishes = "dishes" in spark.table(f"{silver}.stg_restaurant_mentions").columns

# Transformation

Dish category: first match on the unaccented dish name.

In [ ]:
DISH_CATEGORIES = [   # first match on the unaccented dish name
    ("Đồ uống", r"\\b(tra|ca phe|cafe|coffee|nuoc|sinh to|sua|bia|ruou|soda|tra sua|matcha|cacao|nuoc ep)\\b"),
    ("Tráng miệng", r"\\b(che|kem|flan|rau cau|tau hu|banh flan|yaourt|sua chua|banh ngot|tiramisu)\\b"),
    ("Món nước", r"\\b(pho|bun|hu tieu|mi|banh canh|chao|lau|sup|mien|banh da|cao lau|nui)\\b"),
    ("Cơm / xôi / bánh mì", r"\\b(com|xoi|banh mi|com tam|com chien)\\b"),
    ("Hải sản", r"\\b(oc|tom|cua|ghe|so|muc|ca|bach tuoc|hau|ngheu|hen|tom hum)\\b"),
    ("Nướng / chiên", r"\\b(nuong|chien|quay|bbq|xien|ran)\\b"),
    ("Ăn vặt", r"\\b(banh trang|goi cuon|cha gio|nem|bot chien|banh xeo|banh khot|xien que|banh bot loc)\\b"),
]
category_rule = " ".join(f"WHEN dish_plain RLIKE '{rx}' THEN '{cat}'" for cat, rx in DISH_CATEGORIES)
dishes = spark.sql(f"""    WITH d AS (
        SELECT m.restaurant_id, normalize_keyword(x.name) AS dish_name_normalized, trim(x.name) AS dish,
               CASE WHEN x.price_vnd BETWEEN 5000 AND 5000000 THEN x.price_vnd END AS price_vnd,
               coalesce(x.is_signature, false) AS is_signature, m.video_id, m.extracted_at
        FROM mentions_keyed m
        LATERAL VIEW explode({'m.dishes' if has_dishes else 'CAST(NULL AS ARRAY<STRUCT<name: STRING, price_vnd: BIGINT, is_signature: BOOLEAN>>)'}) t AS x
        WHERE m.restaurant_id IS NOT NULL AND length(trim(x.name)) > 1
    ),
    g AS (
        SELECT restaurant_id, dish_name_normalized,
               max_by(dish, length(dish))                    AS dish_name,
               lower(unaccent_vi(dish_name_normalized))      AS dish_plain,
               CAST(percentile_approx(price_vnd, 0.5) AS BIGINT) AS price_vnd,
               MIN(price_vnd)                                AS price_min_vnd,
               MAX(price_vnd)                                AS price_max_vnd,
               bool_or(is_signature)                         AS is_signature,
               COUNT_IF(is_signature)                        AS signature_votes,
               COUNT(DISTINCT video_id)                      AS video_count,
               MIN(extracted_at)                             AS first_seen_at,
               MAX(extracted_at)                             AS last_seen_at
        FROM d GROUP BY restaurant_id, dish_name_normalized
    )
    SELECT g.restaurant_id, g.dish_name_normalized, g.dish_name,
           CASE {category_rule} ELSE 'Khác' END AS dish_category,
           g.price_vnd, g.price_min_vnd, g.price_max_vnd, g.is_signature, CAST(g.signature_votes AS INT) AS signature_votes,
           CAST(g.video_count AS INT) AS video_count, g.first_seen_at, g.last_seen_at,
           'video' AS source
    FROM g
    WHERE EXISTS (SELECT 1 FROM {gold}.dim_restaurant_scd2 r WHERE r.restaurant_id = g.restaurant_id)
""")

result_df = dishes.select(
    F.col("restaurant_id").cast("string").alias("restaurant_id"),
    F.col("dish_name_normalized").cast("string").alias("dish_name_normalized"),
    F.col("dish_name").cast("string").alias("dish_name"),
    F.col("dish_category").cast("string").alias("dish_category"),
    F.col("price_vnd").cast("bigint").alias("price_vnd"),
    F.col("price_min_vnd").cast("bigint").alias("price_min_vnd"),
    F.col("price_max_vnd").cast("bigint").alias("price_max_vnd"),
    F.col("is_signature").cast("boolean").alias("is_signature"),
    F.col("signature_votes").cast("int").alias("signature_votes"),
    F.col("video_count").cast("int").alias("video_count"),
    F.col("first_seen_at").cast("timestamp").alias("first_seen_at"),
    F.col("last_seen_at").cast("timestamp").alias("last_seen_at"),
    F.col("source").cast("string").alias("source"),
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["restaurant_id", "dish_name_normalized"]
result_df = add_checksums(result_df, primary_key)

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        restaurant_id STRING, dish_name_normalized STRING, dish_name STRING, dish_category STRING,
        price_vnd BIGINT, price_min_vnd BIGINT, price_max_vnd BIGINT, is_signature BOOLEAN, signature_votes INT,
        video_count INT, first_seen_at TIMESTAMP, last_seen_at TIMESTAMP, source STRING,
        etl_date TIMESTAMP, etl_by STRING, primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Dish dimension, SCD 1 (poc method): restaurant_id x dish named in the YouTube reviews. Child of dim_restaurant_scd2.'
""")

# Write to Table

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source_df")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source_df AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        WHEN NOT MATCHED BY SOURCE THEN DELETE
    """)

## Check: one row per restaurant × dish

In [ ]:
c = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT restaurant_id) AS restaurants,
           COUNT(*) - COUNT(DISTINCT restaurant_id, dish_name_normalized) AS dup
    FROM {path}""").first()
assert c["dup"] == 0, f"{path}: {c['dup']} duplicate restaurant x dish rows"
print(f"✅ {path}: {c['n']} dishes of {c['restaurants']} restaurants")
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 1").first()
print(last["operation"], dict(last["operationMetrics"] or {}))